# OlpAI NLP: Fast Contest Transformer
Pipeline dịch Hoa → Việt theo bộ khung lời giải Top-5, nhưng từng module được viết lại bằng PyTorch cơ bản. Chọn T4 rồi **Run all**; mặc định train thật toàn bộ dữ liệu.

## 1. Cài thư viện

In [ ]:
%pip install -q sentencepiece sacrebleu tqdm pandas gdown

## 2. Tải code, dữ liệu và cấu hình
Cell này tự tải mọi thứ cần thiết. `SMOKE_TEST=False` là bản thi thật; chỉ đổi thành `True` khi cần kiểm tra nhanh.

In [ ]:
import os, shutil, subprocess, sys, time, zipfile
from pathlib import Path

# Repo chỉ chứa code. Dữ liệu được tải từ link Drive BTC đã cung cấp.
REPO_URL = 'https://github.com/TrDuy-pan3000/olpai-nlp-fast.git'
PROJECT_DIR = Path('/content/olpai-nlp-fast')
DATA_ROOT = Path('/content/olpai-data')
DATA_DIR = DATA_ROOT / 'dataset'
ARTIFACT_DIR = Path('/content/olpai-artifacts')

if not (PROJECT_DIR / 'nlp_basic.py').exists():
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(PROJECT_DIR)], check=True)
else:
    subprocess.run(['git', '-C', str(PROJECT_DIR), 'pull', '--ff-only'], check=True)

required_file = DATA_DIR / 'train/train.zh'
if not required_file.exists():
    import gdown
    DATA_ROOT.mkdir(parents=True, exist_ok=True)
    zip_path = DATA_ROOT / 'dataset.zip'
    gdown.download(id='190wIN301_X2Z7dqtPmLA7Z4Cggn3twl4', output=str(zip_path), quiet=False)
    with zipfile.ZipFile(zip_path) as archive:
        archive.extractall(DATA_ROOT)
assert required_file.exists(), f'Không tìm thấy dữ liệu tại {required_file}'

ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(PROJECT_DIR))

SMOKE_TEST = False # False = train thật toàn bộ dữ liệu; True chỉ để debug nhanh
USE_BEAM = False   # Luôn tạo greedy submission trước; beam chỉ bật khi còn thời gian
RESUME = False     # True: tiếp tục nếu last_model.pt vẫn còn trong runtime
SEED = 42

import math, torch
import torch.nn as nn
import torch.nn.functional as F
from copy import deepcopy
from dataclasses import dataclass
sys.modules.pop('nlp_basic', None)  # tránh Colab giữ code cũ sau git pull
from nlp_basic import (
    PAD_ID, BOS_ID, EOS_ID, TranslationDataset, causal_mask,
    seed_everything, read_lines, read_parallel_data, deduplicate_pairs, group_split,
    build_translation_memory, train_sentencepiece, filter_pairs_by_token_length,
    make_loader, make_optimizer, WarmupInverseSqrtScheduler, train_one_epoch,
    evaluate_loss, evaluate_bleu, save_checkpoint, load_checkpoint,
    translate_sentences, beam_search_decode_sentence, write_submission_csv,
    package_submission, package_tokenizer,
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
if device.type == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.set_float32_matmul_precision('high')
elif not SMOKE_TEST:
    raise RuntimeError('Fast Contest cần GPU T4. Chọn Runtime > Change runtime type > T4 GPU.')
seed_everything(SEED)
print('CHẾ ĐỘ:', 'SMOKE TEST (không có chất lượng)' if SMOKE_TEST else 'FULL CONTEST TRAINING')

## 3. Đọc, kiểm tra và chia dữ liệu

In [ ]:
TRAIN_ZH = DATA_DIR / 'train/train.zh'
TRAIN_VI = DATA_DIR / 'train/train.vi'
PUBLIC_ZH = DATA_DIR / 'public_test/public_test.zh'
PRIVATE_ZH = DATA_DIR / 'private_test/private_test.zh'

raw_pairs = read_parallel_data(TRAIN_ZH, TRAIN_VI)
memory = build_translation_memory([x[0] for x in raw_pairs], [x[1] for x in raw_pairs])
pairs = deduplicate_pairs(raw_pairs)
train_pairs, valid_pairs = group_split(pairs, valid_ratio=0.05, seed=SEED)
assert {s for s, _ in train_pairs}.isdisjoint({s for s, _ in valid_pairs})

if SMOKE_TEST:
    train_pairs = train_pairs[:1024]
    valid_pairs = valid_pairs[:256]

print(f'Raw: {len(raw_pairs):,} | Sau dedup: {len(pairs):,}')
print(f'Train: {len(train_pairs):,} | Valid: {len(valid_pairs):,}')
print('Ví dụ:', train_pairs[0])

## 4. Huấn luyện joint SentencePiece BPE

In [ ]:
VOCAB_SIZE = 2000 if SMOKE_TEST else 8000
tokenizer = train_sentencepiece(train_pairs, ARTIFACT_DIR, vocab_size=VOCAB_SIZE)
train_pairs = filter_pairs_by_token_length(train_pairs, tokenizer, max_len=40)
valid_pairs = filter_pairs_by_token_length(valid_pairs, tokenizer, max_len=40)
print(f'Sau lọc độ dài | Train: {len(train_pairs):,} | Valid: {len(valid_pairs):,}')
print('Vocab thực tế:', tokenizer.get_piece_size())
print('Token mẫu:', tokenizer.encode(train_pairs[0][0], out_type=str))

## 5. Kiến trúc Transformer basic viết tường minh
Luồng chính: `Embedding + Position → Encoder → Decoder → Linear`. Các cell dưới đây tương ứng trực tiếp với các khối trong lời giải mẫu.

### 5.1. Cấu hình và positional encoding
Thay `RopeConfig/RoPE` bằng config thường và sinusoidal position dễ đọc. Tensor luôn có dạng `[batch, sequence, d_model]`.

In [ ]:
@dataclass
class ModelConfig:
    vocab_size: int = 8000
    d_model: int = 256
    nhead: int = 8
    encoder_layers: int = 4
    decoder_layers: int = 4
    dim_feedforward: int = 1024
    dropout: float = 0.1
    max_len: int = 40

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len, dropout):
        super().__init__()
        position = torch.arange(max_len).unsqueeze(1)
        frequency = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
        pe = torch.zeros(max_len, d_model)
        pe[:, 0::2] = torch.sin(position * frequency)
        pe[:, 1::2] = torch.cos(position * frequency)
        self.register_buffer('pe', pe.unsqueeze(0), persistent=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.dropout(x + self.pe[:, :x.size(1)])

### 5.2. Multi-head attention và feed-forward
Thay `GroupedQueryAttentionRoPE` bằng attention chuẩn; thay `FFN_SwiGLU` bằng hai lớp Linear với ReLU.

In [ ]:
class BasicMultiHeadAttention(nn.Module):
    def __init__(self, d_model, nhead, dropout):
        super().__init__()
        self.attention = nn.MultiheadAttention(d_model, nhead, dropout=dropout, batch_first=True)

    def forward(self, query, key, value, key_padding_mask=None, attn_mask=None):
        output, _ = self.attention(query, key, value, key_padding_mask=key_padding_mask,
                                   attn_mask=attn_mask, need_weights=False)
        return output

class BasicFeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout):
        super().__init__()
        self.layers = nn.Sequential(nn.Linear(d_model, d_ff), nn.ReLU(),
                                    nn.Dropout(dropout), nn.Linear(d_ff, d_model))

    def forward(self, x):
        return self.layers(x)

### 5.3. Encoder
Mỗi layer: self-attention → residual + LayerNorm → feed-forward → residual + LayerNorm.

In [ ]:
class EncoderLayer(nn.Module):
    def __init__(self, d_model, nhead, d_ff, dropout):
        super().__init__()
        self.self_attention = BasicMultiHeadAttention(d_model, nhead, dropout)
        self.feed_forward = BasicFeedForward(d_model, d_ff, dropout)
        self.norm1, self.norm2 = nn.LayerNorm(d_model), nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, padding_mask=None):
        attended = self.self_attention(x, x, x, key_padding_mask=padding_mask)
        x = self.norm1(x + self.dropout(attended))
        transformed = self.feed_forward(x)
        return self.norm2(x + self.dropout(transformed))

class Encoder(nn.Module):
    def __init__(self, layer, num_layers, d_model):
        super().__init__()
        self.layers = nn.ModuleList(deepcopy(layer) for _ in range(num_layers))
        self.final_norm = nn.LayerNorm(d_model)

    def forward(self, x, padding_mask=None):
        for layer in self.layers:
            x = layer(x, padding_mask)
        return self.final_norm(x)

### 5.4. Decoder
Decoder có đủ masked self-attention, cross-attention nhìn output encoder và feed-forward.

In [ ]:
class DecoderLayer(nn.Module):
    def __init__(self, d_model, nhead, d_ff, dropout):
        super().__init__()
        self.self_attention = BasicMultiHeadAttention(d_model, nhead, dropout)
        self.cross_attention = BasicMultiHeadAttention(d_model, nhead, dropout)
        self.feed_forward = BasicFeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, memory, tgt_padding_mask=None, memory_padding_mask=None):
        future_mask = causal_mask(x.size(1), x.device)
        attended = self.self_attention(x, x, x, tgt_padding_mask, future_mask)
        x = self.norm1(x + self.dropout(attended))
        crossed = self.cross_attention(x, memory, memory, memory_padding_mask)
        x = self.norm2(x + self.dropout(crossed))
        transformed = self.feed_forward(x)
        return self.norm3(x + self.dropout(transformed))

class Decoder(nn.Module):
    def __init__(self, layer, num_layers, d_model):
        super().__init__()
        self.layers = nn.ModuleList(deepcopy(layer) for _ in range(num_layers))
        self.final_norm = nn.LayerNorm(d_model)

    def forward(self, x, memory, tgt_padding_mask=None, memory_padding_mask=None):
        for layer in self.layers:
            x = layer(x, memory, tgt_padding_mask, memory_padding_mask)
        return self.final_norm(x)

### 5.5. Ghép TransformerModel hoàn chỉnh
Embedding dùng chung với output projection (weight tying) để giảm tham số.

In [ ]:
class Seq2SeqTransformer(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.embedding = nn.Embedding(config.vocab_size, config.d_model, padding_idx=PAD_ID)
        self.position = PositionalEncoding(config.d_model, config.max_len, config.dropout)
        self.encoder = Encoder(EncoderLayer(config.d_model, config.nhead,
                                            config.dim_feedforward, config.dropout),
                               config.encoder_layers, config.d_model)
        self.decoder = Decoder(DecoderLayer(config.d_model, config.nhead,
                                            config.dim_feedforward, config.dropout),
                               config.decoder_layers, config.d_model)
        self.output = nn.Linear(config.d_model, config.vocab_size, bias=False)
        self.output.weight = self.embedding.weight
        self.scale = math.sqrt(config.d_model)
        nn.init.normal_(self.embedding.weight, mean=0.0, std=config.d_model ** -0.5)
        with torch.no_grad():
            self.embedding.weight[PAD_ID].zero_()

    def encode(self, src):
        src_padding = src.eq(PAD_ID)
        hidden = self.position(self.embedding(src) * self.scale)
        return self.encoder(hidden, src_padding), src_padding

    def decode(self, tgt, memory, src_padding):
        tgt_padding = tgt.eq(PAD_ID)
        hidden = self.position(self.embedding(tgt) * self.scale)
        hidden = self.decoder(hidden, memory, tgt_padding, src_padding)
        return self.output(hidden)

    def forward(self, src, tgt_input):
        memory, src_padding = self.encode(src)
        return self.decode(tgt_input, memory, src_padding)

### 5.6. Các khối phụ có trong lời giải mẫu
Dataset hai chiều và contrastive được giữ dưới dạng basic để học, nhưng để `False/0.0` khi thi nhằm không tăng thời gian. Label smoothing và beam hypothesis được dùng theo cùng bộ khung.

In [ ]:
class BidirectionalTranslationDataset(TranslationDataset):
    def __init__(self, pairs, tokenizer, max_len=40, include_reverse=False):
        expanded = list(pairs)
        if include_reverse:
            expanded += [(target, source) for source, target in pairs]
        super().__init__(expanded, tokenizer, max_len)

class LabelSmoothedCrossEntropyLoss(nn.Module):
    def __init__(self, ignore_index=PAD_ID, smoothing=0.1):
        super().__init__()
        self.ignore_index, self.smoothing = ignore_index, smoothing

    def forward(self, logits, targets):
        return F.cross_entropy(logits, targets, ignore_index=self.ignore_index,
                               label_smoothing=self.smoothing)

@dataclass
class BeamSearchHypothesis:
    tokens: list[int]
    score: float

@dataclass
class ContrastiveConfig:
    d_model: int = 256
    projection_dim: int = 128
    temperature: float = 0.1
    weight: float = 0.0

class ProjectionHead(nn.Module):
    def __init__(self, d_model, projection_dim):
        super().__init__()
        self.layers = nn.Sequential(nn.Linear(d_model, d_model), nn.ReLU(),
                                    nn.Linear(d_model, projection_dim))
    def forward(self, x):
        return F.normalize(self.layers(x), dim=-1)

def mean_pool(hidden, padding_mask):
    valid = (~padding_mask).unsqueeze(-1).to(hidden.dtype)
    return (hidden * valid).sum(1) / valid.sum(1).clamp_min(1.0)

def contrastive_loss(source_vectors, target_vectors, temperature=0.1):
    source_vectors = F.normalize(source_vectors, dim=-1)
    target_vectors = F.normalize(target_vectors, dim=-1)
    scores = source_vectors @ target_vectors.T / temperature
    labels = torch.arange(scores.size(0), device=scores.device)
    return (F.cross_entropy(scores, labels) + F.cross_entropy(scores.T, labels)) / 2

def compute_crosslingual_loss(model, projection, src, tgt, temperature=0.1):
    source_hidden, source_pad = model.encode(src)
    target_hidden, target_pad = model.encode(tgt)
    source_vector = projection(mean_pool(source_hidden, source_pad))
    target_vector = projection(mean_pool(target_hidden, target_pad))
    return contrastive_loss(source_vector, target_vector, temperature)

def select_vi2zh_window(epoch, start_epoch=3, end_epoch=5):
    return start_epoch <= epoch <= end_epoch

def contrastive_train_epoch(model, projection, loader, optimizer, device, temperature=0.1):
    model.train(); projection.train(); total = 0.0
    for src, tgt in tqdm(loader, desc='Contrastive', leave=False):
        src, tgt = src.to(device), tgt.to(device)
        optimizer.zero_grad(set_to_none=True)
        loss = compute_crosslingual_loss(model, projection, src, tgt, temperature)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(list(model.parameters()) + list(projection.parameters()), 1.0)
        optimizer.step(); total += loss.item()
    return total / max(1, len(loader))

class WarmupInverseSqrtScheduler:
    def __init__(self, optimizer, peak_lr=5e-4, warmup_steps=300):
        self.optimizer, self.peak_lr = optimizer, peak_lr
        self.warmup_steps, self.step_num = warmup_steps, 0

    def step(self):
        self.step_num += 1
        if self.step_num <= self.warmup_steps:
            lr = self.peak_lr * self.step_num / self.warmup_steps
        else:
            lr = self.peak_lr * math.sqrt(self.warmup_steps / self.step_num)
        for group in self.optimizer.param_groups:
            group['lr'] = lr
        return lr

    def state_dict(self):
        return {'peak_lr': self.peak_lr, 'warmup_steps': self.warmup_steps, 'step_num': self.step_num}

    def load_state_dict(self, state):
        self.peak_lr, self.warmup_steps = state['peak_lr'], state['warmup_steps']
        self.step_num = state['step_num']
        if self.step_num:
            lr = self.peak_lr * (self.step_num / self.warmup_steps if self.step_num <= self.warmup_steps
                                 else math.sqrt(self.warmup_steps / self.step_num))
            for group in self.optimizer.param_groups:
                group['lr'] = lr

INCLUDE_REVERSE = False       # module có đủ nhưng không bật khi thi
CONTRASTIVE_WEIGHT = 0.0      # module có đủ nhưng không tăng thời gian train

## 6. DataLoader, loss, optimizer và model

In [ ]:
if SMOKE_TEST:
    config = ModelConfig(vocab_size=tokenizer.get_piece_size(), d_model=96, nhead=4,
                         encoder_layers=1, decoder_layers=1, dim_feedforward=192,
                         dropout=0.1, max_len=32)
    BATCH_SIZE, EPOCHS, WARMUP, EVAL_EVERY = 64, 1, 20, 1
else:
    config = ModelConfig(vocab_size=tokenizer.get_piece_size(), d_model=256, nhead=8,
                         encoder_layers=4, decoder_layers=4, dim_feedforward=1024,
                         dropout=0.1, max_len=40)
    BATCH_SIZE, EPOCHS, WARMUP, EVAL_EVERY = 128, 15, 300, 2

train_loader = make_loader(train_pairs, tokenizer, BATCH_SIZE, config.max_len, True, 2)
valid_loader = make_loader(valid_pairs, tokenizer, BATCH_SIZE, config.max_len, False, 2)
model = Seq2SeqTransformer(config).to(device)
criterion = LabelSmoothedCrossEntropyLoss(ignore_index=PAD_ID, smoothing=0.1)
optimizer = make_optimizer(model, lr=5e-4)
scheduler = WarmupInverseSqrtScheduler(optimizer, peak_lr=5e-4, warmup_steps=WARMUP)
scaler = torch.amp.GradScaler('cuda', enabled=device.type == 'cuda')
print(f'Tham số: {sum(p.numel() for p in model.parameters()):,}')

## 7. Kiểm tra một batch

In [ ]:
src, tgt = next(iter(train_loader))
with torch.no_grad():
    logits = model(src.to(device), tgt[:, :-1].to(device))
assert logits.shape[:2] == tgt[:, 1:].shape
assert logits.shape[-1] == config.vocab_size
print('Smoke shape OK:', tuple(logits.shape))
del logits
if device.type == 'cuda': torch.cuda.empty_cache()

## 8. Huấn luyện
Fast Contest thường mất khoảng 8–18 phút train trên T4. BLEU chỉ chạy mỗi 2 epoch để tiết kiệm thời gian.

In [ ]:
best_bleu, bad_evals, start_epoch = -1.0, 0, 1
last_path = ARTIFACT_DIR / 'last_model.pt'
if RESUME and last_path.exists():
    state = load_checkpoint(last_path, model, optimizer, scheduler, scaler, device)
    start_epoch = state['epoch'] + 1
    best_bleu = state['best_bleu']
    print(f'Resume từ epoch {state["epoch"]}, best BLEU={best_bleu:.2f}')
start_time = time.time()
for epoch in range(start_epoch, EPOCHS + 1):
    train_loss = train_one_epoch(model, train_loader, optimizer, scheduler, criterion,
                                 device, scaler, epoch)
    val_loss = evaluate_loss(model, valid_loader, criterion, device)
    should_eval = (epoch % EVAL_EVERY == 0) or epoch == EPOCHS or SMOKE_TEST
    bleu = evaluate_bleu(model, valid_loader, tokenizer, device, config.max_len) if should_eval else None

    save_checkpoint(ARTIFACT_DIR / 'last_model.pt', model, optimizer, scheduler,
                    epoch, best_bleu, config, scaler)
    if bleu is not None:
        if bleu > best_bleu:
            best_bleu, bad_evals = bleu, 0
            save_checkpoint(ARTIFACT_DIR / 'best_model.pt', model, optimizer, scheduler,
                            epoch, best_bleu, config, scaler)
        else:
            bad_evals += 1
    print(f'Epoch {epoch:02d} | train={train_loss:.3f} | val={val_loss:.3f} | '
          f'BLEU={bleu if bleu is not None else "skip"} | best={best_bleu:.2f}')
    if bad_evals >= 3:
        print('Early stopping: BLEU không tăng sau 3 lần đánh giá.')
        break
print(f'Tổng thời gian train: {(time.time()-start_time)/60:.1f} phút')

## 9. Xem nhanh một số bản dịch validation

In [ ]:
load_checkpoint(ARTIFACT_DIR / 'best_model.pt', model, device=device)
sample_src = [s for s, _ in valid_pairs[:10]]
sample_ref = [t for _, t in valid_pairs[:10]]
sample_pred = translate_sentences(model, sample_src, tokenizer, device, batch_size=10, max_len=config.max_len)
import pandas as pd
display(pd.DataFrame({'Hoa': sample_src, 'Tham chiếu': sample_ref, 'Mô hình': sample_pred}))

## 10. Tạo submission
Greedy theo batch là mặc định nhanh. Chỉ bật `USE_BEAM=True` nếu đã có submission greedy an toàn và còn thời gian.

In [ ]:
from tqdm.auto import tqdm
public_src, private_src = read_lines(PUBLIC_ZH), read_lines(PRIVATE_ZH)

def make_predictions(sentences):
    if not USE_BEAM:
        return translate_sentences(model, sentences, tokenizer, device, memory,
                                   batch_size=BATCH_SIZE, max_len=config.max_len)
    output = []
    for sentence in tqdm(sentences, desc='Beam-3 inference'):
        key = normalize_text(sentence)
        output.append(memory[key] if key in memory else
                      beam_search_decode_sentence(model, key, tokenizer, device, 3, config.max_len, 0.6))
    return output

public_pred = make_predictions(public_src)
private_pred = make_predictions(private_src)
public_csv = write_submission_csv(public_src, public_pred, ARTIFACT_DIR / 'public_test.csv')
private_csv = write_submission_csv(private_src, private_pred, ARTIFACT_DIR / 'private_test.csv')
submission = package_submission(public_csv, private_csv, ARTIFACT_DIR / 'submission.zip')
print('Đã tạo:', submission)

# Colab tự bật hộp tải file kết quả về máy.
from google.colab import files
files.download(str(submission))

## 11. Chiến thuật phòng thi
1. Chạy smoke test. 2. Chuyển `SMOKE_TEST=False`, chọn T4 và Run all. 3. Tạo greedy submission ngay. 4. Chỉ thử beam-3 hoặc tăng epoch nếu còn thời gian. Hãy tải `submission.zip` và checkpoint về máy trước khi đóng runtime.